# 04 — Baseline models

Six classical algorithms trained on both experiments and scored on the same held-out test set.

**Pipeline module:** `src/train.py`

*Every number and figure below was produced by the pipeline in `src/`. This
notebook reads those results; it does not re-implement them.*

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Image, display

TABLES = ROOT / "results" / "tables"
FIGURES = ROOT / "results" / "figures"

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

def table(name, **kw):
    """Read one of the pipeline's result tables."""
    return pd.read_csv(TABLES / name, **kw)

def figure(name):
    """Display one of the pipeline's figures."""
    return Image(filename=str(FIGURES / name))

print(f"project root: {ROOT}")

project root: C:\Users\babu\OneDrive\Desktop\ML Project


## The six algorithms

| Model | Type |
|---|---|
| Logistic Regression | linear baseline |
| KNN | distance-based |
| Decision Tree | single non-linear tree |
| Random Forest | tree ensemble (bagging) |
| SVM (RBF) | margin-based |
| XGBoost | tree ensemble (boosting) |

Scale-sensitive models (Logistic Regression, KNN, SVM) are wrapped in a `Pipeline`
with `StandardScaler`, so scaling is fitted on training folds only and never sees the
test set.

**Two models train on reduced sets**, recorded in the results table: SVM on 30,000
rows and KNN on 50,000, because their cost grows with sample size. Both still predict
on the full 80,008-row test set, so the comparison stays fair where it matters.

Reproduce with:

```bash
python src/train.py
```

In [2]:
binary = table("baseline_models_binary.csv")
binary

,model,experiment,train_rows,accuracy,precision,recall,f1,roc_auc,pr_auc,train_seconds,predict_seconds
0,XGBoost,A (binary),320032,0.998975,0.995976,0.998021,0.996997,0.999969,0.999843,7.94,0.16
1,Random Forest,A (binary),320032,0.998763,0.996262,0.996481,0.996372,0.999957,0.999782,28.85,0.25
2,Decision Tree,A (binary),320032,0.998275,0.993856,0.996041,0.994947,0.997654,0.992005,9.24,0.02
3,KNN,A (binary),50000,0.982577,0.947853,0.950077,0.948964,0.992204,0.976184,0.06,8.11
4,SVM (RBF),A (binary),30000,0.955717,0.980034,0.755663,0.853347,NaN,NaN,13.27,56.76
5,Logistic Regression,A (binary),320032,0.919596,0.913492,0.583682,0.712260,0.961840,0.896879,4.21,0.04


In [3]:
multi = table("baseline_models_multi.csv")
multi

,model,experiment,train_rows,accuracy,precision,recall,f1,roc_auc,pr_auc,train_seconds,predict_seconds
0,XGBoost,B (multi),320032,0.998813,0.977810,0.944294,0.958281,0.999973,0.987474,38.59,0.57
1,Random Forest,B (multi),320032,0.998725,0.987635,0.935692,0.958251,0.990413,0.961006,29.64,0.34
2,Decision Tree,B (multi),320032,0.998250,0.976064,0.941934,0.956328,0.972577,0.927174,7.10,0.02
3,KNN,B (multi),50000,0.982252,0.688557,0.625079,0.649362,0.944862,0.750004,0.12,8.34
4,Logistic Regression,B (multi),320032,0.942531,0.636769,0.463530,0.477933,0.967855,0.575989,66.48,0.03
5,SVM (RBF),B (multi),30000,0.956329,0.429819,0.357031,0.384435,NaN,NaN,7.19,30.33


## Why SVM's AUC columns are blank

ROC-AUC and PR-AUC need a score per class that behaves like a probability. `SVC`
without `probability=True` produces one-vs-one voting margins, which do not sum to 1.
Computing an AUC from them would report a number the metric does not apply to, so the
columns are left empty and the reason documented. Enabling `probability=True` would
trigger an internal cross-validation that makes an already-slow model roughly five
times slower.

## What the results show

The tree family wins decisively on both experiments, and the reason is structural
rather than incidental. "Is this an attack?" is not a smooth boundary in feature
space — it is a set of conditions. *If packet sizes are unusually variable, and the
timing is machine-regular, and the destination port is unusual, then PortScan.*
Trees represent rules natively; Logistic Regression and SVM have to approximate them
with a surface.